In [ ]:
#### This will run according to the settins mentioned in 
#### technology stack settings.doc under laptop 1 section


In [ ]:
#### Cell_0 this code is used to generate the claude_master file


#Requirements: pip install openpyxl

from collections import Counter, defaultdict
from pathlib import Path
from openpyxl import Workbook, load_workbook

INPUT_DIR = Path(r"D:\BORUE")  # EDIT this directory
PILOT_FILE = INPUT_DIR / "BRUSE_Brave_Evaluated.xlsx"
PHASE2_FILE = INPUT_DIR / "BRUSE_Phase2_Common_vs_Query2.xlsx"
QWEN_FILE = INPUT_DIR / "Qwen_Brave_Evaluated.xlsx"
OUTPUT_FILE = INPUT_DIR / "BRUSE_Claude_Validation_Master.xlsx"

SOURCE_HEADER = [
    "common", "condition", "expanded_query", "brave_rank", "url", "title",
    "snippet", "llm_grade", "relevance_label", "judge_reason",
    "input_excel_row", "submitted_query", "query_truncated"
]
TARGET_HEADER = [
    "concept", "condition", "expanded_query", "brave_rank", "url", "title",
    "snippet", "llm_grade", "relevance_label", "judge_reason",
    "input_excel_row", "submitted_query", "query_truncated", "source_dataset"
]


def copy_ranked_results(source_path, destination, label, condition_map, counts, concepts):
    if not source_path.is_file():
        raise FileNotFoundError(f"Missing input: {source_path}")
    book = load_workbook(source_path, read_only=True, data_only=True)
    try:
        if "Ranked_Results" not in book.sheetnames:
            raise ValueError(f"Missing Ranked_Results in {source_path.name}")
        sheet = book["Ranked_Results"]
        iterator = sheet.iter_rows(values_only=True)
        header = list(next(iterator))
        if header[:13] != SOURCE_HEADER:
            raise ValueError(f"Unexpected source columns in {source_path.name}: {header}")
        for excel_row, values in enumerate(iterator, start=2):
            condition = values[1]
            if condition not in condition_map:
                continue  # Pilot excludes query4/query6/full
            if len(values) < 13:
                values = (*values, *(None for _ in range(13 - len(values))))
            if len(values) != 13:
                raise ValueError(f"Unexpected width in {source_path.name}, row {excel_row}")
            mapped = condition_map[condition]
            destination.append((values[0], mapped, *values[2:13], label))
            counts[mapped] += 1
            concepts[mapped].add(int(values[10]))  # IDs distinguish duplicate spellings
    finally:
        book.close()


def main():
    if OUTPUT_FILE.resolve() in [p.resolve() for p in (PILOT_FILE, PHASE2_FILE, QWEN_FILE)]:
        raise ValueError("Output path must not overwrite an input")
    wb = Workbook(write_only=True)
    master = wb.create_sheet("Master_Results")
    master.append(TARGET_HEADER)
    counts = Counter()
    concepts = defaultdict(set)

    
    copy_ranked_results(PILOT_FILE, master, "Pilot",
                        {"common": "Common", "bruse_query2": "BRUSE"},
                        counts, concepts)
    copy_ranked_results(PHASE2_FILE, master, "Phase2",
                        {"common": "Common", "bruse_query2": "BRUSE"},
                        counts, concepts)
    copy_ranked_results(QWEN_FILE, master, "Qwen",
                        {"qwen_expanded": "Qwen"},
                        counts, concepts)

    expected_counts = {"Common": 24410, "BRUSE": 24410, "Qwen": 24410}
    if dict(counts) != expected_counts:
        raise ValueError(f"Unexpected URL counts: {dict(counts)}")
    for condition in expected_counts:
        if len(concepts[condition]) != 2441:
            raise ValueError(f"Unexpected concept count for {condition}: {len(concepts[condition])}")
    if not (concepts["Common"] == concepts["BRUSE"] == concepts["Qwen"]):
        raise ValueError("Condition concept ID sets differ")

    audit = wb.create_sheet("Audit")
    audit.append(["Condition", "Concepts", "URL judgments", "Expected"])
    for condition in ("Common", "BRUSE", "Qwen"):
        audit.append([condition, 2441, counts[condition], 24410])
    audit.append(["TOTAL", 2441, sum(counts.values()), 73230])
    audit.append([])
    audit.append(["Source", "Included conditions"])
    audit.append(["Pilot", "Common; BRUSE"])
    audit.append(["Phase II", "Common; BRUSE"])
    audit.append(["Qwen", "Qwen"])

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    wb.save(OUTPUT_FILE)
    print(f"Created: {OUTPUT_FILE}")
    print(f"Master_Results: {sum(counts.values())} records; Audit: 10 rows")
    print("Conditions:", dict(counts))


if __name__ == "__main__":
    main()


In [ ]:
### Cell_1 this code is used to create 1000 rows from 
### combined master file. it will be used in inter-rater agreement

import pandas as pd
import numpy as np
from pathlib import Path



input_file = Path(r"D:\BORUE\BRUSE_Claude_Validation_Master.xlsx")
output_file = Path(r"D:\BORUE\Claude_Validation_Sample_1000.xlsx")

SEED = 20261114
TARGETS = {"Common": 334, "bruse_query2": 333, "Qwen": 333}



df = pd.read_excel(input_file, sheet_name="Master_Results")
df.columns = df.columns.str.strip()

required = [
    "concept", "condition", "expanded_query",
    "brave_rank", "url", "title", "snippet",
    "llm_grade", "input_excel_row", "source_dataset"
]

missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if len(df) != 73230:
    raise ValueError(f"Expected 73230 rows; found {len(df)}")


def normalize_condition(value):
    s = str(value).strip().lower()
    if s == "common":
        return "Common"
    if s in ("bruse_query2", "bruse", "bruse_query2"):
        return "bruse_query2"
    if s.startswith("qwen"):
        return "Qwen"
    raise ValueError(f"Unknown condition: {value}")

df["condition"] = df["condition"].map(normalize_condition)
df["llm_grade"] = pd.to_numeric(df["llm_grade"], errors="raise")
df["brave_rank"] = pd.to_numeric(df["brave_rank"], errors="raise")

assert df["llm_grade"].isin([0, 1, 2, 3]).all()
assert df["brave_rank"].between(1, 10).all()

for condition in TARGETS:
    count = (df["condition"] == condition).sum()
    assert count == 24410, f"{condition}: {count} rows"


df["master_excel_row"] = np.arange(2, len(df) + 2)



rng = np.random.default_rng(SEED)
selected_parts = []
audit_rows = []

for condition, target in TARGETS.items():

    part = df[df["condition"] == condition].copy()


    strata = {
        key: group.index.to_numpy()
        for key, group in part.groupby(
            ["llm_grade", "brave_rank"], sort=True
        )
    }


    allocation = {key: 0 for key in strata}

    remaining = target
    active = sorted(strata)

    while remaining > 0:
        eligible = [
            key for key in active
            if allocation[key] < len(strata[key])
        ]

        if not eligible:
            raise ValueError("Insufficient records for sampling")

        for key in eligible:
            if remaining == 0:
                break
            allocation[key] += 1
            remaining -= 1

    for key in active:
        population = len(strata[key])
        n = allocation[key]

        if n == 0:
            continue

        chosen = rng.choice(
            strata[key], size=n, replace=False
        )

        selected_parts.append(df.loc[chosen].copy())

        audit_rows.append({
            "condition": condition,
            "original_grade": key[0],
            "brave_rank": key[1],
            "population_N": population,
            "sample_n": n,
            "inclusion_probability": n / population,
            "analysis_weight": population / n
        })

sample = pd.concat(selected_parts, ignore_index=True)
audit = pd.DataFrame(audit_rows)

# ============================================================
# 3. ASSIGN WEIGHTS AND BLIND VALIDATION INPUT
# ============================================================

sample = sample.merge(
    audit,
    left_on=["condition", "llm_grade", "brave_rank"],
    right_on=["condition", "original_grade", "brave_rank"],
    how="left",
    validate="many_to_one"
)

# Shuffle selected records independently of their original order
sample = sample.iloc[rng.permutation(len(sample))].reset_index(drop=True)
sample.insert(
    0, "sample_id",
    [f"GV{i:04d}" for i in range(1, len(sample) + 1)]
)


claude_columns = [
    "sample_id", "concept", "condition",
    "expanded_query", "brave_rank", "url",
    "title", "snippet"
]

# Include reference translation when available.
# Do not substitute expanded_query as the intended meaning.
reference_candidates = [
    "English_Translated", "english_translated",
    "reference_translation"
]

reference_col = next(
    (c for c in reference_candidates if c in sample.columns),
    None
)

if reference_col:
    claude_columns.insert(2, reference_col)

claude_input = sample[claude_columns].copy()
claude_input["claude_grade"] = ""
claude_input["claude_reason"] = ""

original_columns = [
    "sample_id", "master_excel_row",
    "source_dataset", "input_excel_row",
    "concept", "condition", "brave_rank",
    "llm_grade", "relevance_label",
    "judge_reason", "inclusion_probability",
    "analysis_weight"
]

original_columns = [
    c for c in original_columns if c in sample.columns
]

original_judgments = sample[original_columns].copy()

# ============================================================
# 4. SAVE OUTPUT
# ============================================================

output_file.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    claude_input.to_excel(
        writer, sheet_name="claude_Input", index=False
    )
    original_judgments.to_excel(
        writer, sheet_name="Original_Judgments", index=False
    )
    audit.to_excel(
        writer, sheet_name="Sampling_Audit", index=False
    )

# ============================================================
# 5. VERIFY
# ============================================================

assert len(sample) == 1000
assert sample["master_excel_row"].is_unique
assert sample.groupby("condition").size().to_dict() == TARGETS
assert sample["analysis_weight"].notna().all()

print("SAMPLING COMPLETED")
print("=" * 55)
print("Master records:", len(df))
print("Selected records:", len(sample))
print("Random seed:", SEED)

print("\nCondition distribution:")
print(sample["condition"].value_counts().sort_index())

print("\nOriginal relevance grade distribution:")
print(pd.crosstab(sample["condition"], sample["llm_grade"]))

print("\nRank distribution:")
print(pd.crosstab(sample["condition"], sample["brave_rank"]))

print("\nSaved:", output_file)


In [ ]:
%pip install -U google-genai openpyxl pandas

In [ ]:
# installed due to incompatibility issue with previous cell
%pip install --force-reinstall "pandas==2.3.3"

In [ ]:
%pip install --no-deps "python-dateutil==2.9.0.post0" "numpy==2.3.5"

In [ ]:
import pandas as pd
import numpy as np
import dateutil
from google import genai

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("python-dateutil:", dateutil.__version__)
print("claude SDK: OK")

In [ ]:
%pip install --no-deps --upgrade anthropic

In [ ]:
%pip install --no-deps jiter

In [ ]:
import anthropic
import pandas as pd

print("Anthropic SDK:", anthropic.__version__)
print("Pandas:", pd.__version__)
print("Anthropic SDK imported successfully.")

In [ ]:
%pip check

In [ ]:
%pip install --no-deps docstring-parser

In [ ]:
import anthropic
import pandas as pd

print("Anthropic SDK:", anthropic.__version__)
print("Pandas:", pd.__version__)
print("Anthropic SDK: READY")

In [ ]:

import os
import getpass
import anthropic

# ============================================================
# 1. SECURE API KEY
# ============================================================

api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    api_key = getpass.getpass(
        "Enter your Anthropic API key: "
    ).strip()

if not api_key:
    raise ValueError("Anthropic API key is missing.")

# ============================================================
# 2. CREATE CLIENT
# ============================================================

client = anthropic.Anthropic(
    api_key=api_key,
    timeout=60.0,
    max_retries=2
)

# ============================================================
# 3. LIST AVAILABLE SONNET MODELS
# ============================================================

print("Checking Anthropic API connectivity...")
print("=" * 60)

try:
    models = client.models.list(limit=100)

    sonnet_models = [
        model for model in models.data
        if "sonnet" in model.id.lower()
    ]

    if sonnet_models:
        print("Available Claude Sonnet models:\n")

        for model in sonnet_models:
            print("Model ID:", model.id)
            print("Display name:", model.display_name)
            print("-" * 60)
    else:
        print("No Sonnet models found in this response.")
        print("Check your account's available models.")

except anthropic.AuthenticationError:
    print("Authentication failed. Check your API key.")

except anthropic.PermissionDeniedError:
    print("Access denied. Check your project permissions.")

except anthropic.RateLimitError:
    print("Rate limit or quota restriction encountered.")

except anthropic.APIError as e:
    print("Anthropic API error:", str(e))


In [ ]:

import requests
import socket

HOST = "api.anthropic.com"

print("ANTHROPIC NETWORK DIAGNOSTIC")
print("=" * 55)

# Test 1: DNS resolution
try:
    ip = socket.gethostbyname(HOST)
    print("DNS resolution: PASS")
    print("Resolved IP:", ip)
except Exception as e:
    print("DNS resolution: FAILED")
    print("Details:", str(e))

print("-" * 55)

# Test 2: HTTPS connectivity
try:
    response = requests.get(
        "https://api.anthropic.com/v1/models",
        timeout=20
    )

    print("HTTPS connection: ESTABLISHED")
    print("HTTP status:", response.status_code)

    if response.status_code in (401, 403):
        print(
            "The API server responded. "
            "Authentication was not supplied in this test."
        )
    else:
        print("Response preview:", response.text[:250])

except requests.exceptions.RequestException as e:
    print("HTTPS connection: FAILED")
    print("Error type:", type(e).__name__)
    print("Details:", str(e))


In [ ]:

import os
import getpass
import requests

# Reuse the key if already available in this kernel.
api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    api_key = getpass.getpass(
        "Enter Anthropic API key: "
    ).strip()

url = "https://api.anthropic.com/v1/models"

headers = {
    "x-api-key": api_key,
    "anthropic-version": "2023-06-01"
}

try:
    response = requests.get(
        url,
        headers=headers,
        params={"limit": 100},
        timeout=30
    )

    print("HTTP status:", response.status_code)

    if response.status_code == 200:
        data = response.json()
        models = data.get("data", [])

        print("\nAUTHENTICATION SUCCESSFUL")
        print("\nAvailable Sonnet models:")

        for model in models:
            model_id = model.get("id", "")
            if "sonnet" in model_id.lower():
                print(
                    model_id,
                    "|",
                    model.get("display_name", "")
                )

    else:
        print("API response:")
        print(response.text[:1000])

except requests.exceptions.RequestException as e:
    print("Connection failed:", type(e).__name__)
    print(str(e))


In [ ]:
### Cell_2
### 5 query test with claude ###
import os
import json
import getpass
import requests
import pandas as pd

# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_FILE = (
    r"D:\BORUE\Claude_Validation_Sample_1000.xlsx"
)

SHEET_NAME = "Claude_Input"
MODEL_ID = "claude-sonnet-5-5"
PILOT_SIZE = 5

# ============================================================
# 2. LOAD SAMPLE
# ============================================================

df = pd.read_excel(
    INPUT_FILE,
    sheet_name=SHEET_NAME,
    dtype={"sample_id": str}
)

required_columns = [
    "sample_id",
    "concept",
    "url",
    "title",
    "snippet"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

if len(df) < PILOT_SIZE:
    raise ValueError("Insufficient sample records.")

pilot = df.head(PILOT_SIZE).copy()

if pilot["sample_id"].isna().any():
    raise ValueError("Missing sample IDs.")

if pilot["sample_id"].duplicated().any():
    raise ValueError("Duplicate sample IDs.")

# ============================================================
# 3. PREPARE DOCUMENTS
# ============================================================

documents = []

for _, row in pilot.iterrows():

    def clean(value):
        if pd.isna(value):
            return ""
        return str(value).strip()

    documents.append({
        "id": clean(row["sample_id"]),
        "query": clean(row["concept"]),
        "url": clean(row["url"]),
        "title": clean(row["title"]),
        "snippet": clean(row["snippet"])
    })

# ============================================================
# 4. ORIGINAL RELEVANCE ASSESSMENT RUBRIC
# ============================================================

SYSTEM_PROMPT = """You are an independent information retrieval relevance assessor.

Judge each document ONLY against the original Roman Urdu
query, not against any expanded query.

Use only its title, snippet, and URL.

Relevance grades:

3 = Highly relevant: directly addresses the intended concept.

2 = Relevant: substantially related and useful.

1 = Marginally relevant: limited relationship.

0 = Not relevant: unrelated or misleading.

Roman Urdu can be ambiguous. Do not assume that phonetically
similar terms necessarily have the same meaning.

Return JSON:

{
  "judgments": [
    {"id": "1", "grade": 2, "reason": "Brief explanation"}
  ]
}

Return exactly one judgment for EVERY supplied document ID.

Use the IDs exactly as provided.

Do not invent additional IDs."""

# ============================================================
# 5. API AUTHENTICATION
# ============================================================

api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    api_key = getpass.getpass(
        "Enter Anthropic API key: "
    ).strip()

if not api_key:
    raise ValueError("Anthropic API key is missing.")

# ============================================================
# 6. CALL CLAUDE
# ============================================================

payload = {
    "model": MODEL_ID,
    "max_tokens": 2000,
    "system": SYSTEM_PROMPT,
    "messages": [
        {
            "role": "user",
            "content": (
                "Assess the following documents. "
                "Return only the requested JSON object.\n\n"
                + json.dumps(
                    {"documents": documents},
                    ensure_ascii=False
                )
            )
        }
    ]
}

headers = {
    "x-api-key": api_key,
    "anthropic-version": "2023-06-01",
    "content-type": "application/json"
}

print("Submitting five records to Claude Sonnet 5.5...")

response = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers=headers,
    json=payload,
    timeout=180
)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    raise RuntimeError(
        f"Claude API error: {response.text[:1500]}"
    )

api_result = response.json()

# ============================================================
# 7. EXTRACT RESPONSE
# ============================================================

if api_result.get("stop_reason") != "end_turn":
    raise ValueError(
        "Claude did not complete normally. "
        f"Stop reason: {api_result.get('stop_reason')}"
    )

response_text = "\n".join(
    block["text"]
    for block in api_result.get("content", [])
    if block.get("type") == "text"
).strip()

if response_text.startswith("```"):
    lines = response_text.splitlines()
    if lines[-1].strip() == "```":
        response_text = "\n".join(lines[1:-1])
        response_text = response_text.strip()

if response_text.lower().startswith("json\n"):
    response_text = response_text[5:].strip()

parsed = json.loads(response_text)

judgments = parsed["judgments"]

# ============================================================
# 8. STRICT VALIDATION
# ============================================================

expected_ids = {
    str(doc["id"]) for doc in documents
}

returned_ids = [
    str(item["id"]) for item in judgments
]

if len(judgments) != PILOT_SIZE:
    raise ValueError(
        "Claude returned an incorrect number of judgments."
    )

if len(set(returned_ids)) != len(returned_ids):
    raise ValueError("Duplicate judgment IDs returned.")

if set(returned_ids) != expected_ids:
    raise ValueError(
        "Returned IDs do not match submitted IDs."
    )

for item in judgments:

    grade = item.get("grade")

    if (
        type(grade) is not int
        or grade not in (0, 1, 2, 3)
    ):
        raise ValueError(
            f"Invalid relevance grade: {grade}"
        )

    if not isinstance(item.get("reason"), str):
        raise ValueError("Invalid or missing reason.")

# ============================================================
# 9. DISPLAY PILOT RESULTS
# ============================================================

result_df = pd.DataFrame([
    {
        "sample_id": str(item["id"]),
        "claude_grade": item["grade"],
        "claude_reason": item["reason"]
    }
    for item in judgments
])

result_df = (
    pilot[["sample_id", "concept"]]
    .merge(
        result_df,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

print("\nCLAUDE PILOT RESULTS")
print("=" * 65)

display(result_df)

usage = api_result.get("usage", {})

print("\nAPI USAGE")
print("Input tokens:", usage.get("input_tokens"))
print("Output tokens:", usage.get("output_tokens"))

print("\nPilot completed successfully.")
print("Original Excel workbook unchanged.")


In [ ]:
### Cell_3 (just check point)

import os
import json
import time
import getpass
import requests
import pandas as pd

# ============================================================
# 1. FILE PATHS AND CONFIGURATION
# ============================================================

WORK_DIR = (
    r"D:\BORUE"
)

INPUT_FILE = os.path.join(WORK_DIR,"claude_Validation_Sample_1000.xlsx")

OUTPUT_FILE = os.path.join(WORK_DIR,"Claude_Validation_Checkpoint.xlsx")

MODEL_ID = "claude-sonnet-5-5"
BATCH_SIZE = 10
MAX_RETRIES = 3

# ============================================================
# 2. LOAD ORIGINAL SAMPLE
# ============================================================

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="claude_Input",
    dtype={"sample_id": str}
)

required_columns = [
    "sample_id",
    "concept",
    "url",
    "title",
    "snippet"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

if df["sample_id"].isna().any():
    raise ValueError("Missing sample IDs.")

if df["sample_id"].duplicated().any():
    raise ValueError("Duplicate sample IDs.")

if len(df) != 1000:
    raise ValueError(
        f"Expected 1000 records, found {len(df)}."
    )

df["sample_id"] = df["sample_id"].astype(str)

# ============================================================
# 3. LOAD EXISTING CHECKPOINT IF AVAILABLE
# ============================================================

result_columns = [
    "sample_id",
    "claude_grade",
    "claude_reason"
]

if os.path.exists(OUTPUT_FILE):

    completed = pd.read_excel(
        OUTPUT_FILE,
        sheet_name="Claude_Judgments",
        dtype={"sample_id": str}
    )

    print(
        "Existing checkpoint loaded:",
        len(completed),
        "records"
    )

else:

    completed = pd.DataFrame(
        columns=result_columns
    )

completed_ids = set(
    completed["sample_id"].astype(str)
)

pending = df[
    ~df["sample_id"].isin(completed_ids)
].copy()

if pending.empty:
    print("All records already completed.")
    raise SystemExit

# Process only the first 10 pending records
batch = pending.head(BATCH_SIZE).copy()

# ============================================================
# 4. PREPARE DOCUMENTS
# ============================================================

def clean(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

documents = []

for _, row in batch.iterrows():

    documents.append({
        "id": clean(row["sample_id"]),
        "query": clean(row["concept"]),
        "url": clean(row["url"]),
        "title": clean(row["title"]),
        "snippet": clean(row["snippet"])
    })

# ============================================================
# 5. ORIGINAL RELEVANCE RUBRIC
# ============================================================

SYSTEM_PROMPT = """You are an independent information retrieval relevance assessor.

Judge each document ONLY against the original Roman Urdu
query, not against any expanded query.

Use only its title, snippet, and URL.

Relevance grades:

3 = Highly relevant: directly addresses the intended concept.

2 = Relevant: substantially related and useful.

1 = Marginally relevant: limited relationship.

0 = Not relevant: unrelated or misleading.

Roman Urdu can be ambiguous. Do not assume that phonetically
similar terms necessarily have the same meaning.

Return JSON:

{
  "judgments": [
    {"id": "1", "grade": 2, "reason": "Brief explanation"}
  ]
}

Return exactly one judgment for EVERY supplied document ID.

Use the IDs exactly as provided.

Do not invent additional IDs."""

# ============================================================
# 6. API KEY
# ============================================================

api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    api_key = getpass.getpass(
        "Enter Anthropic API key: "
    ).strip()

if not api_key:
    raise ValueError("Missing Anthropic API key.")

headers = {
    "x-api-key": api_key,
    "anthropic-version": "2023-06-01",
    "content-type": "application/json"
}

# ============================================================
# 7. SUBMIT BATCH WITH RETRIES
# ============================================================

payload = {
    "model": MODEL_ID,
    "max_tokens": 4000,
    "system": SYSTEM_PROMPT,
    "messages": [
        {
            "role": "user",
            "content": (
                "Assess the following documents. "
                "Return only the requested JSON object.\n\n"
                + json.dumps(
                    {"documents": documents},
                    ensure_ascii=False
                )
            )
        }
    ]
}

api_result = None

for attempt in range(1, MAX_RETRIES + 1):

    try:
        print(
            f"Submitting batch of {len(documents)} records "
            f"(attempt {attempt})..."
        )

        response = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers=headers,
            json=payload,
            timeout=180
        )

        print("HTTP status:", response.status_code)

        if response.status_code == 200:
            api_result = response.json()
            break

        if response.status_code not in (
            408, 429, 500, 502, 503, 504, 529
        ):
            raise RuntimeError(
                response.text[:1500]
            )

        print(
            "Temporary API error:",
            response.text[:500]
        )

    except requests.exceptions.RequestException as e:
        print("Connection error:", str(e))

    if attempt < MAX_RETRIES:
        wait_seconds = 5 * (2 ** (attempt - 1))
        print(
            f"Waiting {wait_seconds} seconds..."
        )
        time.sleep(wait_seconds)

if api_result is None:
    raise RuntimeError(
        "Batch failed after maximum retries."
    )

# ============================================================
# 8. PARSE AND VALIDATE RESPONSE
# ============================================================

if api_result.get("stop_reason") != "end_turn":
    raise ValueError(
        "Incomplete Claude response: "
        + str(api_result.get("stop_reason"))
    )

response_text = "\n".join(
    block["text"]
    for block in api_result.get("content", [])
    if block.get("type") == "text"
).strip()

if response_text.startswith("```"):
    lines = response_text.splitlines()
    if lines[-1].strip() == "```":
        response_text = "\n".join(lines[1:-1]).strip()

if response_text.lower().startswith("json\n"):
    response_text = response_text[5:].strip()

parsed = json.loads(response_text)
judgments = parsed["judgments"]

expected_ids = {
    d["id"] for d in documents
}

returned_ids = [
    str(j["id"]) for j in judgments
]

if len(judgments) != len(documents):
    raise ValueError(
        "Incorrect number of judgments."
    )

if len(set(returned_ids)) != len(returned_ids):
    raise ValueError(
        "Duplicate judgment IDs."
    )

if set(returned_ids) != expected_ids:
    raise ValueError(
        "Judgment IDs do not match."
    )

for judgment in judgments:

    grade = judgment.get("grade")

    if (
        type(grade) is not int
        or grade not in (0, 1, 2, 3)
    ):
        raise ValueError(
            f"Invalid grade: {grade}"
        )

    if not isinstance(
        judgment.get("reason"), str
    ):
        raise ValueError(
            "Invalid judgment reason."
        )

# ============================================================
# 9. APPEND AND SAVE CHECKPOINT
# ============================================================

new_results = pd.DataFrame([
    {
        "sample_id": str(j["id"]),
        "claude_grade": j["grade"],
        "claude_reason": j["reason"]
    }
    for j in judgments
])

updated = pd.concat(
    [completed, new_results],
    ignore_index=True
)

if updated["sample_id"].duplicated().any():
    raise ValueError(
        "Duplicate IDs in checkpoint."
    )

# Save through a temporary workbook first.
temp_file = os.path.join(
    WORK_DIR,
    "Claude_Validation_Checkpoint.tmp.xlsx"
)

with pd.ExcelWriter(
    temp_file,
    engine="openpyxl"
) as writer:

    updated.to_excel(
        writer,
        sheet_name="Claude_Judgments",
        index=False
    )

os.replace(temp_file, OUTPUT_FILE)

# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("\nBATCH COMPLETED SUCCESSFULLY")
print("=" * 60)

display(
    batch[["sample_id", "concept"]].merge(
        new_results,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )
)

usage = api_result.get("usage", {})

print("\nMODEL:", MODEL_ID)
print("Input tokens:", usage.get("input_tokens"))
print("Output tokens:", usage.get("output_tokens"))

print("\nCheckpoint saved:", OUTPUT_FILE)
print("Total completed:", len(updated))
print("Remaining:", len(df) - len(updated))
print("Original sample workbook: UNCHANGED")


In [ ]:
### Cell_4 1000 query test

import os
import json
import time
import random
import getpass
import requests
import pandas as pd

# ============================================================
# 1. CONFIGURATION
# ============================================================

WORK_DIR = (
    r"D:\BORUE"    
)

INPUT_FILE = os.path.join(
    WORK_DIR,
    "claude_Validation_Sample_1000.xlsx"
)

OUTPUT_FILE = os.path.join(
    WORK_DIR,
    "Claude_Validation_Checkpoint.xlsx"
)

TEMP_FILE = os.path.join(
    WORK_DIR,
    "Claude_Validation_Checkpoint.tmp.xlsx"
)

MODEL_ID = "claude-sonnet-5-5"

BATCH_SIZE = 10
MAX_RETRIES = 5
REQUEST_TIMEOUT = 180
BATCH_DELAY = 2



df = pd.read_excel(
    INPUT_FILE,
    sheet_name="claude_Input",
    dtype={"sample_id": str}
)

required_columns = [
    "sample_id",
    "concept",
    "url",
    "title",
    "snippet"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing input columns: {missing}"
    )

if len(df) != 1000:
    raise ValueError(
        f"Expected 1000 records, found {len(df)}."
    )

if df["sample_id"].isna().any():
    raise ValueError("Missing sample IDs.")

df["sample_id"] = df["sample_id"].astype(str)

if df["sample_id"].duplicated().any():
    raise ValueError("Duplicate sample IDs.")

all_ids = set(df["sample_id"])

print("Original sample loaded:", len(df))



result_columns = [
    "sample_id",
    "claude_grade",
    "claude_reason"
]

if os.path.exists(OUTPUT_FILE):

    completed = pd.read_excel(
        OUTPUT_FILE,
        sheet_name="Claude_Judgments",
        dtype={"sample_id": str}
    )

    print("Existing checkpoint:", len(completed))

else:

    completed = pd.DataFrame(
        columns=result_columns
    )

for col in result_columns:
    if col not in completed.columns:
        raise ValueError(
            f"Missing checkpoint column: {col}"
        )

if completed["sample_id"].isna().any():
    raise ValueError("Checkpoint contains missing IDs.")

completed["sample_id"] = (
    completed["sample_id"].astype(str)
)

if completed["sample_id"].duplicated().any():
    raise ValueError(
        "Duplicate IDs in existing checkpoint."
    )

unknown_ids = (
    set(completed["sample_id"]) - all_ids
)

if unknown_ids:
    raise ValueError(
        f"Unknown checkpoint IDs: {unknown_ids}"
    )

for _, row in completed.iterrows():

    grade = row["claude_grade"]

    if (
        pd.isna(grade)
        or float(grade) not in (0, 1, 2, 3)
    ):
        raise ValueError(
            "Invalid grade in existing checkpoint."
        )

    if (
        pd.isna(row["claude_reason"])
        or not str(row["claude_reason"]).strip()
    ):
        raise ValueError(
            "Missing reason in existing checkpoint."
        )

completed_ids = set(completed["sample_id"])

pending = df[
    ~df["sample_id"].isin(completed_ids)
].copy()

print("Remaining records:", len(pending))



SYSTEM_PROMPT = """You are an independent information retrieval relevance assessor.

Judge each document ONLY against the original Roman Urdu
query, not against any expanded query.

Use only its title, snippet, and URL.

Relevance grades:

3 = Highly relevant: directly addresses the intended concept.

2 = Relevant: substantially related and useful.

1 = Marginally relevant: limited relationship.

0 = Not relevant: unrelated or misleading.

Roman Urdu can be ambiguous. Do not assume that phonetically
similar terms necessarily have the same meaning.

Return JSON:

{
  "judgments": [
    {"id": "1", "grade": 2, "reason": "Brief explanation"}
  ]
}

Return exactly one judgment for EVERY supplied document ID.

Use the IDs exactly as provided.

Do not invent additional IDs."""



api_key = os.environ.get("ANTHROPIC_API_KEY")

if pending.empty:
    print("All 1000 records already completed.")

else:

    if not api_key:
        api_key = getpass.getpass(
            "Enter Anthropic API key: "
        ).strip()

    if not api_key:
        raise ValueError(
            "Anthropic API key is missing."
        )

HEADERS = {
    "x-api-key": api_key or "",
    "anthropic-version": "2023-06-01",
    "content-type": "application/json"
}



def clean(value):

    if pd.isna(value):
        return ""

    return str(value).strip()


def prepare_documents(batch):

    documents = []

    for _, row in batch.iterrows():

        documents.append({
            "id": clean(row["sample_id"]),
            "query": clean(row["concept"]),
            "url": clean(row["url"]),
            "title": clean(row["title"]),
            "snippet": clean(row["snippet"])
        })

    return documents


def parse_and_validate(api_result, documents):

    if api_result.get("stop_reason") != "end_turn":
        raise ValueError(
            "Incomplete response. Stop reason: "
            + str(api_result.get("stop_reason"))
        )

    content_blocks = api_result.get("content", [])

    response_text = "\n".join(
        block.get("text", "")
        for block in content_blocks
        if block.get("type") == "text"
    ).strip()

    if response_text.startswith("```"):

        lines = response_text.splitlines()

        if (
            len(lines) >= 2
            and lines[-1].strip() == "```"
        ):
            response_text = "\n".join(
                lines[1:-1]
            ).strip()

    if response_text.lower().startswith("json\n"):
        response_text = response_text[5:].strip()

    parsed = json.loads(response_text)

    if not isinstance(parsed, dict):
        raise ValueError(
            "Response is not a JSON object."
        )

    judgments = parsed.get("judgments")

    if not isinstance(judgments, list):
        raise ValueError(
            "Missing judgments array."
        )

    expected_ids = {
        doc["id"] for doc in documents
    }

    returned_ids = []

    for judgment in judgments:

        if not isinstance(judgment, dict):
            raise ValueError(
                "Invalid judgment object."
            )

        judgment_id = str(
            judgment.get("id", "")
        )

        returned_ids.append(judgment_id)

        grade = judgment.get("grade")

        if (
            type(grade) is not int
            or grade not in (0, 1, 2, 3)
        ):
            raise ValueError(
                f"Invalid grade for {judgment_id}"
            )

        reason = judgment.get("reason")

        if (
            not isinstance(reason, str)
            or not reason.strip()
        ):
            raise ValueError(
                f"Missing reason for {judgment_id}"
            )

    if len(judgments) != len(documents):
        raise ValueError(
            "Incorrect judgment count."
        )

    if len(set(returned_ids)) != len(returned_ids):
        raise ValueError(
            "Duplicate returned IDs."
        )

    if set(returned_ids) != expected_ids:
        raise ValueError(
            "Returned IDs do not match submitted IDs."
        )

    return pd.DataFrame([
        {
            "sample_id": str(j["id"]),
            "claude_grade": j["grade"],
            "claude_reason": j["reason"]
        }
        for j in judgments
    ])


def save_checkpoint(results):

    if results["sample_id"].duplicated().any():
        raise ValueError(
            "Duplicate IDs before checkpoint save."
        )

    with pd.ExcelWriter(
        TEMP_FILE,
        engine="openpyxl"
    ) as writer:

        results.to_excel(
            writer,
            sheet_name="Claude_Judgments",
            index=False
        )

    os.replace(TEMP_FILE, OUTPUT_FILE)


def submit_batch(documents):

    payload = {
        "model": MODEL_ID,
        "max_tokens": 4000,
        "system": SYSTEM_PROMPT,
        "messages": [
            {
                "role": "user",
                "content": (
                    "Assess the following documents. "
                    "Return only the requested JSON object.\n\n"
                    + json.dumps(
                        {"documents": documents},
                        ensure_ascii=False
                    )
                )
            }
        ]
    }

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = requests.post(
                "https://api.anthropic.com/v1/messages",
                headers=HEADERS,
                json=payload,
                timeout=REQUEST_TIMEOUT
            )

            if response.status_code == 200:

                api_result = response.json()

                # Validate before accepting the batch.
                new_results = parse_and_validate(
                    api_result,
                    documents
                )

                return new_results, api_result

            if response.status_code not in (
                408, 429, 500, 502, 503, 504, 529
            ):
                raise RuntimeError(
                    f"Non-retryable HTTP "
                    f"{response.status_code}: "
                    f"{response.text[:1000]}"
                )

            print(
                f"Temporary HTTP {response.status_code}"
            )

            retry_after = response.headers.get(
                "retry-after"
            )

        except (
            requests.exceptions.RequestException,
            ValueError,
            KeyError,
            TypeError
        ) as error:

            print(
                f"Attempt {attempt} failed: {error}"
            )

            retry_after = None

        if attempt == MAX_RETRIES:
            break

        wait_seconds = min(
            60,
            5 * (2 ** (attempt - 1))
        )

        if retry_after:

            try:
                wait_seconds = max(
                    wait_seconds,
                    float(retry_after)
                )
            except ValueError:
                pass

        wait_seconds += random.uniform(0, 1)

        print(
            f"Retrying after {wait_seconds:.1f} seconds..."
        )

        time.sleep(wait_seconds)

    raise RuntimeError(
        "Batch failed after maximum retries."
    )



total_input_tokens = 0
total_output_tokens = 0
successful_batches = 0

if not pending.empty:

    total_batches = (
        len(pending) + BATCH_SIZE - 1
    ) // BATCH_SIZE

    print("\nStarting remaining evaluation")
    print("Model:", MODEL_ID)
    print("Pending batches:", total_batches)
    print("=" * 65)

    for start in range(
        0,
        len(pending),
        BATCH_SIZE
    ):

        batch_number = (
            start // BATCH_SIZE
        ) + 1

        batch = pending.iloc[
            start:start + BATCH_SIZE
        ].copy()

        documents = prepare_documents(batch)

        print(
            f"\nBatch {batch_number}/{total_batches}"
            f" | Records: {len(documents)}"
        )

        try:

            new_results, api_result = submit_batch(
                documents
            )

            updated = pd.concat(
                [completed, new_results],
                ignore_index=True
            )

            if updated["sample_id"].duplicated().any():
                raise ValueError(
                    "Duplicate IDs after merging."
                )

            if not set(updated["sample_id"]).issubset(
                all_ids
            ):
                raise ValueError(
                    "Unknown IDs after merging."
                )


            save_checkpoint(updated)

            completed = updated

            usage = api_result.get("usage", {})

            input_tokens = (
                usage.get("input_tokens") or 0
            )

            output_tokens = (
                usage.get("output_tokens") or 0
            )

            total_input_tokens += input_tokens
            total_output_tokens += output_tokens

            successful_batches += 1

            print(
                "SAVED | Completed:",
                len(completed),
                "/ 1000"
            )

            print(
                "Batch tokens:",
                input_tokens,
                "input,",
                output_tokens,
                "output"
            )

        except Exception as error:

            print("\nPROCESSING STOPPED SAFELY")
            print("Failed batch:", batch_number)
            print("Error:", str(error))
            print(
                "Previously saved judgments:",
                len(completed)
            )
            print(
                "Checkpoint:",
                OUTPUT_FILE
            )
            print(
                "Rerun this cell to resume."
            )

            break

        if batch_number < total_batches:
            time.sleep(BATCH_DELAY)



remaining_count = (
    len(df) - len(completed)
)

print("\n" + "=" * 65)
print("CLAUDE EVALUATION SUMMARY")
print("=" * 65)

print("Model:", MODEL_ID)
print("Total sample:", len(df))
print("Completed:", len(completed))
print("Remaining:", remaining_count)

print(
    "Successful batches this run:",
    successful_batches
)

print(
    "Input tokens this run:",
    total_input_tokens
)

print(
    "Output tokens this run:",
    total_output_tokens
)

print("Checkpoint:", OUTPUT_FILE)

if remaining_count == 0:
    print("\nALL 1000 JUDGMENTS COMPLETED.")
else:
    print(
        "\nEvaluation incomplete. "
        "Rerun this cell to resume."
    )

print("Original sample workbook: UNCHANGED")


In [ ]:
#### Cell_5 Analysis starts from here
#Generate Claude_GPT_Agreement_Analysis.xlsx


import os
import numpy as np
import pandas as pd



BASE_DIR = (r"D:\BORUE")

SAMPLE_FILE = os.path.join(BASE_DIR,"claude_Validation_Sample_1000.xlsx")

CLAUDE_FILE = os.path.join(BASE_DIR,"Claude_Validation_Checkpoint.xlsx")

OUTPUT_FILE = os.path.join(BASE_DIR,"Claude_GPT_Agreement_Analysis.xlsx")

BOOTSTRAP_REPS = 1200
RANDOM_SEED = 20261114



original = pd.read_excel(
    SAMPLE_FILE,
    sheet_name="Original_Judgments",
    dtype={"sample_id": str}
)

sample_input = pd.read_excel(
    SAMPLE_FILE,
    sheet_name="claude_Input",
    dtype={"sample_id": str}
)

claude = pd.read_excel(
    CLAUDE_FILE,
    sheet_name="Claude_Judgments",
    dtype={"sample_id": str}
)

audit = pd.read_excel(
    SAMPLE_FILE,
    sheet_name="Sampling_Audit"
)

print("Original judgments:", len(original))
print("Claude judgments:", len(claude))
print("Sampling strata:", len(audit))



for name, frame in [
    ("Original", original),
    ("Claude", claude),
    ("Sample input", sample_input)
]:
    if frame["sample_id"].isna().any():
        raise ValueError(
            f"{name} contains missing sample IDs."
        )

    if frame["sample_id"].duplicated().any():
        raise ValueError(
            f"{name} contains duplicate sample IDs."
        )

if not (
    len(original) == 1000
    and len(claude) == 1000
    and len(sample_input) == 1000
):
    raise ValueError(
        "Expected exactly 1000 records in each input."
    )

original_ids = set(original["sample_id"])
claude_ids = set(claude["sample_id"])
input_ids = set(sample_input["sample_id"])

if not (
    original_ids == claude_ids == input_ids
):
    raise ValueError(
        "Sample IDs do not match across files."
    )



matched = original.merge(
    claude[
        ["sample_id", "claude_grade", "claude_reason"]
    ],
    on="sample_id",
    how="inner",
    validate="one_to_one"
)

matched = matched.merge(
    sample_input[
        ["sample_id", "url", "title", "snippet"]
    ],
    on="sample_id",
    how="inner",
    validate="one_to_one"
)

matched["llm_grade"] = pd.to_numeric(
    matched["llm_grade"],
    errors="raise"
)

matched["claude_grade"] = pd.to_numeric(
    matched["claude_grade"],
    errors="raise"
)

matched["analysis_weight"] = pd.to_numeric(
    matched["analysis_weight"],
    errors="raise"
)

matched["brave_rank"] = pd.to_numeric(
    matched["brave_rank"],
    errors="raise"
)

for column in ["llm_grade", "claude_grade"]:
    if not matched[column].isin([0, 1, 2, 3]).all():
        raise ValueError(
            f"Invalid relevance grades in {column}."
        )

if (
    matched["analysis_weight"].isna().any()
    or (matched["analysis_weight"] <= 0).any()
):
    raise ValueError(
        "Missing or invalid sampling weights."
    )

if not matched["brave_rank"].between(1, 10).all():
    raise ValueError("Invalid Brave ranks.")

if len(matched) != 1000:
    raise ValueError("Merge integrity failure.")



required_audit = [
    "condition",
    "original_grade",
    "brave_rank",
    "population_N",
    "sample_n",
    "analysis_weight"
]

if not set(required_audit).issubset(audit.columns):
    raise ValueError(
        "Sampling audit columns are incomplete."
    )

audit["population_N"] = pd.to_numeric(
    audit["population_N"]
)

audit["sample_n"] = pd.to_numeric(
    audit["sample_n"]
)

audit["analysis_weight"] = pd.to_numeric(
    audit["analysis_weight"]
)

if audit.duplicated(
    ["condition", "original_grade", "brave_rank"]
).any():
    raise ValueError(
        "Duplicate sampling strata."
    )

audit_check = audit[
    [
        "condition",
        "original_grade",
        "brave_rank",
        "population_N",
        "sample_n",
        "analysis_weight"
    ]
].copy()

audit_check = audit_check.rename(
    columns={
        "analysis_weight": "audit_weight"
    }
)

matched = matched.merge(
    audit_check,
    left_on=[
        "condition",
        "llm_grade",
        "brave_rank"
    ],
    right_on=[
        "condition",
        "original_grade",
        "brave_rank"
    ],
    how="left",
    validate="many_to_one"
)

if matched["audit_weight"].isna().any():
    raise ValueError(
        "Some records have no matching sampling stratum."
    )

if not np.allclose(
    matched["analysis_weight"],
    matched["audit_weight"],
    rtol=1e-9,
    atol=1e-9
):
    raise ValueError(
        "Sampling weights disagree with audit."
    )

actual_strata = (
    matched.groupby(
        ["condition", "llm_grade", "brave_rank"]
    )
    .size()
    .rename("actual_n")
    .reset_index()
)

audit_counts = audit.merge(
    actual_strata,
    left_on=[
        "condition",
        "original_grade",
        "brave_rank"
    ],
    right_on=[
        "condition",
        "llm_grade",
        "brave_rank"
    ],
    how="left",
    validate="one_to_one"
)

if not (
    audit_counts["sample_n"]
    == audit_counts["actual_n"]
).all():
    raise ValueError(
        "Sampling stratum counts do not match."
    )

population_size = int(
    audit["population_N"].sum()
)

if population_size != 73230:
    raise ValueError(
        f"Unexpected population size: {population_size}"
    )



def weighted_confusion(frame):

    matrix = np.zeros((4, 4), dtype=float)

    gpt = frame["llm_grade"].to_numpy(
        dtype=int
    )

    cla = frame["claude_grade"].to_numpy(
        dtype=int
    )

    weights = frame["analysis_weight"].to_numpy(
        dtype=float
    )

    np.add.at(
        matrix,
        (gpt, cla),
        weights
    )

    return matrix


def calculate_metrics(frame):

    matrix = weighted_confusion(frame)

    total = matrix.sum()

    if total <= 0:
        raise ValueError("Empty weighted matrix.")

    observed = matrix / total

    row_marginal = observed.sum(axis=1)
    col_marginal = observed.sum(axis=0)

    expected = np.outer(
        row_marginal,
        col_marginal
    )

    exact = float(np.trace(observed))

    def kappa(distance):

        observed_disagreement = np.sum(
            distance * observed
        )

        expected_disagreement = np.sum(
            distance * expected
        )

        if expected_disagreement <= 0:
            return np.nan

        return float(
            1 - (
                observed_disagreement
                / expected_disagreement
            )
        )

    grades = np.arange(4)

    linear_distance = (
        np.abs(
            grades[:, None] - grades[None, :]
        ) / 3.0
    )

    quadratic_distance = (
        (
            grades[:, None] - grades[None, :]
        ) ** 2 / 9.0
    )

    linear_kappa = kappa(linear_distance)
    quadratic_kappa = kappa(quadratic_distance)

    both3 = matrix[3, 3]

    gpt3 = matrix[3, :].sum()
    claude3 = matrix[:, 3].sum()

    neither3 = matrix[:3, :3].sum()

    binary_agreement = (
        both3 + neither3
    ) / total

    positive_denominator = (
        gpt3 + claude3
    )

    positive_agreement = (
        2 * both3 / positive_denominator
        if positive_denominator > 0
        else np.nan
    )

    return {
        "Weighted exact": exact,
        "Weighted linear kappa": linear_kappa,
        "Weighted quadratic kappa": quadratic_kappa,
        "Weighted grade3 binary": float(
            binary_agreement
        ),
        "Weighted grade3 positive": float(
            positive_agreement
        ),
        "Weighted joint grade3": float(
            both3 / total
        ),
        "Weighted GPT grade3": float(
            gpt3 / total
        ),
        "Weighted Claude grade3": float(
            claude3 / total
        )
    }



conditions = [
    "All",
    "Common",
    "bruse_query2",
    "Qwen"
]

summary_rows = []

for condition in conditions:

    if condition == "All":
        subset = matched
    else:
        subset = matched[
            matched["condition"] == condition
        ]

    if subset.empty:
        raise ValueError(
            f"No records for {condition}."
        )

    metrics = calculate_metrics(subset)

    display_condition = (
        "BRUSE"
        if condition == "bruse_query2"
        else condition
    )

    summary_rows.append({
        "Condition": display_condition,
        "Sample n": len(subset),
        **metrics
    })

summary_df = pd.DataFrame(summary_rows)



rng = np.random.default_rng(RANDOM_SEED)

strata = [
    group.reset_index(drop=True)
    for _, group in matched.groupby(
        ["condition", "llm_grade", "brave_rank"],
        sort=True
    )
]

metric_names = list(
    calculate_metrics(matched).keys()
)

bootstrap_results = {
    condition: {
        metric: []
        for metric in metric_names
    }
    for condition in conditions
}

print("\nCalculating bootstrap confidence intervals...")

for repetition in range(BOOTSTRAP_REPS):

    resampled_parts = []

    for stratum in strata:

        indices = rng.integers(
            low=0,
            high=len(stratum),
            size=len(stratum)
        )

        resampled_parts.append(
            stratum.iloc[indices]
        )

    bootstrap_sample = pd.concat(
        resampled_parts,
        ignore_index=True
    )

    for condition in conditions:

        if condition == "All":
            subset = bootstrap_sample
        else:
            subset = bootstrap_sample[
                bootstrap_sample["condition"]
                == condition
            ]

        values = calculate_metrics(subset)

        for metric in metric_names:
            bootstrap_results[
                condition
            ][metric].append(
                values[metric]
            )

    if (repetition + 1) % 200 == 0:
        print(
            "Bootstrap repetitions:",
            repetition + 1,
            "/",
            BOOTSTRAP_REPS
        )

ci_rows = []

for condition in conditions:

    subset = (
        matched
        if condition == "All"
        else matched[
            matched["condition"] == condition
        ]
    )

    estimates = calculate_metrics(subset)

    display_condition = (
        "BRUSE"
        if condition == "bruse_query2"
        else condition
    )

    for metric in metric_names:

        values = np.asarray(
            bootstrap_results[condition][metric],
            dtype=float
        )

        valid_values = values[
            np.isfinite(values)
        ]

        if len(valid_values) == 0:
            lower = np.nan
            upper = np.nan
        else:
            lower, upper = np.percentile(
                valid_values,
                [2.5, 97.5]
            )

        ci_rows.append({
            "Condition": display_condition,
            "Metric": metric,
            "Estimate": estimates[metric],
            "CI lower 2.5%": lower,
            "CI upper 97.5%": upper
        })

ci_df = pd.DataFrame(ci_rows)



confusion = pd.crosstab(
    matched["llm_grade"],
    matched["claude_grade"]
)

confusion = confusion.reindex(
    index=[0, 1, 2, 3],
    columns=[0, 1, 2, 3],
    fill_value=0
)

confusion.index.name = "GPT / Claude"

confusion["Total"] = confusion.sum(axis=1)

total_row = confusion.sum(axis=0)
total_row.name = "Total"

confusion = pd.concat([
    confusion,
    total_row.to_frame().T
])

confusion_df = confusion.reset_index()



grade3_rows = []

for condition in conditions:

    subset = (
        matched
        if condition == "All"
        else matched[
            matched["condition"] == condition
        ]
    )

    gpt3 = subset["llm_grade"] == 3
    claude3 = subset["claude_grade"] == 3

    display_condition = (
        "BRUSE"
        if condition == "bruse_query2"
        else condition
    )

    grade3_rows.append({
        "Condition": display_condition,
        "n": len(subset),
        "GPT grade3": int(gpt3.sum()),
        "Claude grade3": int(claude3.sum()),
        "Both grade3": int(
            (gpt3 & claude3).sum()
        ),
        "GPT3 only": int(
            (gpt3 & ~claude3).sum()
        ),
        "Claude3 only": int(
            (~gpt3 & claude3).sum()
        ),
        "Neither grade3": int(
            (~gpt3 & ~claude3).sum()
        )
    })

grade3_df = pd.DataFrame(grade3_rows)



matched_output = pd.DataFrame({
    "Sample ID": matched["sample_id"],
    "Concept": matched["concept"],
    "Condition": matched["condition"].replace(
        {"bruse_query2": "BRUSE"}
    ),
    "Brave rank": matched["brave_rank"],
    "GPT grade": matched["llm_grade"],
    "Claude grade": matched["claude_grade"],
    "Difference": (
        matched["claude_grade"]
        - matched["llm_grade"]
    ),
    "Analysis weight": matched["analysis_weight"],
    "GPT reason": matched["judge_reason"],
    "Claude reason": matched["claude_reason"],
    "URL": matched["url"],
    "Title": matched["title"],
    "Snippet": matched["snippet"]
})



methodology = pd.DataFrame([
    (
        "Input",
        "Original_Judgments and Claude_Judgments "
        "matched one-to-one by sample_id."
    ),
    (
        "Population",
        f"{population_size:,} URL judgments."
    ),
    (
        "Sample",
        "1,000 stratified URL judgments."
    ),
    (
        "Assessors",
        "GPT-4o-mini and Claude Sonnet 5.5."
    ),
    (
        "Relevance scale",
        "Ordinal grades 0, 1, 2, and 3."
    ),
    (
        "Grade 3",
        "Highly relevant."
    ),
    (
        "Sampling weight",
        "Inverse inclusion probability, N_h/n_h."
    ),
    (
        "Stratification",
        "Condition x original GPT grade x Brave rank."
    ),
    (
        "Exact agreement",
        "Weighted proportion of identical grades."
    ),
    (
        "Linear kappa",
        "Cohen's kappa with absolute ordinal "
        "grade-distance disagreement weights."
    ),
    (
        "Quadratic kappa",
        "Cohen's kappa with squared ordinal "
        "grade-distance disagreement weights."
    ),
    (
        "Grade3 binary agreement",
        "Both grade 3 or both below grade 3."
    ),
    (
        "Grade3 positive agreement",
        "2 x weighted both-grade-3 count divided "
        "by the sum of weighted grade-3 counts "
        "from the two assessors."
    ),
    (
        "Bootstrap",
        f"{BOOTSTRAP_REPS} stratified resamples, "
        "sampling URLs with replacement within "
        "each original stratum."
    ),
    (
        "Random seed",
        str(RANDOM_SEED)
    ),
    (
        "Confidence intervals",
        "Percentile 95% bootstrap intervals."
    ),
    (
        "Bootstrap limitation",
        "URL-level resampling does not account "
        "for dependence between URLs belonging "
        "to the same query concept."
    ),
    (
        "Scientific limitation",
        "Cross-model agreement does not establish "
        "ground-truth relevance accuracy."
    )
], columns=["Item", "Details"])



with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    ci_df.to_excel(
        writer,
        sheet_name="Bootstrap_CI",
        index=False
    )

    confusion_df.to_excel(
        writer,
        sheet_name="Confusion_Matrix",
        index=False
    )

    grade3_df.to_excel(
        writer,
        sheet_name="Grade3_Counts",
        index=False
    )

    matched_output.to_excel(
        writer,
        sheet_name="Matched_1000",
        index=False
    )

    methodology.to_excel(
        writer,
        sheet_name="Methodology_Notes",
        index=False
    )


    for sheet in writer.book.worksheets:

        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

        for cell in sheet[1]:
            cell.font = cell.font.copy(
                bold=True
            )

        for column in sheet.columns:

            letter = column[0].column_letter

            max_length = max(
                len(str(cell.value))
                if cell.value is not None
                else 0
                for cell in column[:100]
            )

            sheet.column_dimensions[
                letter
            ].width = min(
                max(max_length + 3, 14),
                55
            )

    for sheet_name, columns in [
        (
            "Summary",
            ["C", "D", "E", "F", "G", "H", "I", "J"]
        ),
        (
            "Bootstrap_CI",
            ["C", "D", "E"]
        )
    ]:
        sheet = writer.book[sheet_name]

        for column in columns:
            for cell in sheet[column][1:]:
                cell.number_format = "0.0000"



print("\n" + "=" * 65)
print("CLAUDE-GPT AGREEMENT ANALYSIS COMPLETED")
print("=" * 65)

print("\nOutput file:")
print(OUTPUT_FILE)

print("\nMatched records:", len(matched))
print("Population size:", population_size)
print("Bootstrap repetitions:", BOOTSTRAP_REPS)

print("\nOVERALL AGREEMENT:")
print(
    summary_df[
        summary_df["Condition"] == "All"
    ].to_string(index=False)
)

print("\nGRADE-3 COUNTS:")
print(
    grade3_df.to_string(index=False)
)

print("\nWORKSHEETS:")
for name in [
    "Summary",
    "Bootstrap_CI",
    "Confusion_Matrix",
    "Grade3_Counts",
    "Matched_1000",
    "Methodology_Notes"
]:
    print(" -", name)

print("\nOriginal input workbooks: UNCHANGED")


In [ ]:
#### Cell_6, overall , cell_2 of analysis

import os
import numpy as np
import pandas as pd

from openpyxl.styles import (
    Font, PatternFill, Alignment
)
from openpyxl.utils import get_column_letter



BASE_DIR = (
    r"D:\BORUE"    
)

INPUT_FILE = os.path.join(BASE_DIR,"Claude_GPT_Agreement_Analysis.xlsx")

OUTPUT_FILE = os.path.join(BASE_DIR,"Claude_GPT_Grade3_Sensitivity_Analysis.xlsx")

BOOTSTRAP_REPS = 2000
RANDOM_SEED = 20261114

CONDITIONS = ["Common", "BRUSE", "Qwen"]



df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Matched_1000",
    dtype={"Sample ID": str}
)

required_columns = [
    "Sample ID",
    "Concept",
    "Condition",
    "Brave rank",
    "GPT grade",
    "Claude grade",
    "Analysis weight"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

if len(df) != 1000:
    raise ValueError(
        f"Expected 1000 records, found {len(df)}"
    )

if (
    df["Sample ID"].isna().any()
    or df["Sample ID"].duplicated().any()
):
    raise ValueError(
        "Missing or duplicate sample IDs."
    )

df["Condition"] = df["Condition"].replace({
    "bruse_query2": "BRUSE"
})

if set(df["Condition"]) != set(CONDITIONS):
    raise ValueError(
        "Unexpected retrieval conditions: "
        + str(df["Condition"].unique())
    )

numeric_columns = [
    "Brave rank",
    "GPT grade",
    "Claude grade",
    "Analysis weight"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="raise"
    )

for column in ["GPT grade", "Claude grade"]:

    if not df[column].isin([0, 1, 2, 3]).all():
        raise ValueError(
            f"Invalid grades in {column}"
        )

if (
    df["Analysis weight"].isna().any()
    or (df["Analysis weight"] <= 0).any()
):
    raise ValueError(
        "Invalid sampling weights."
    )

if not df["Brave rank"].isin(
    range(1, 11)
).all():
    raise ValueError(
        "Invalid Brave result ranks."
    )



df["GPT_grade3"] = (
    df["GPT grade"] == 3
).astype(int)

df["Claude_grade3"] = (
    df["Claude grade"] == 3
).astype(int)

df["Both_grade3"] = (
    (df["GPT_grade3"] == 1)
    & (df["Claude_grade3"] == 1)
).astype(int)

df["Grade3_difference"] = (
    df["Claude_grade3"]
    - df["GPT_grade3"]
)



def weighted_mean(values, weights):

    values = np.asarray(
        values,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    return float(
        np.sum(values * weights)
        / np.sum(weights)
    )


def estimate_rates(data):

    results = {}

    for condition in CONDITIONS:

        subset = data[
            data["Condition"] == condition
        ]

        if subset.empty:
            raise ValueError(
                f"Empty condition: {condition}"
            )

        weights = subset[
            "Analysis weight"
        ].to_numpy()

        gpt_rate = weighted_mean(
            subset["GPT_grade3"],
            weights
        )

        claude_rate = weighted_mean(
            subset["Claude_grade3"],
            weights
        )

        both_rate = weighted_mean(
            subset["Both_grade3"],
            weights
        )

        results[condition] = {
            "GPT": gpt_rate,
            "Claude": claude_rate,
            "Both": both_rate,
            "Change": claude_rate - gpt_rate
        }

    return results


def estimate_contrasts(rates):

    comparisons = [
        ("BRUSE", "Common"),
        ("BRUSE", "Qwen")
    ]

    results = {}

    for a, b in comparisons:

        key = f"{a} - {b}"

        gpt_difference = (
            rates[a]["GPT"]
            - rates[b]["GPT"]
        )

        claude_difference = (
            rates[a]["Claude"]
            - rates[b]["Claude"]
        )

        results[key] = {
            "GPT": gpt_difference,
            "Claude": claude_difference,
            "Change": (
                claude_difference
                - gpt_difference
            )
        }

    return results



rates = estimate_rates(df)

contrasts = estimate_contrasts(rates)

rate_rows = []

for condition in CONDITIONS:

    subset = df[
        df["Condition"] == condition
    ]

    rate_rows.append({
        "Condition": condition,
        "Sample n": len(subset),
        "Estimated population URLs": int(round(
            subset["Analysis weight"].sum()
        )),
        "GPT grade3 rate": rates[condition]["GPT"],
        "Claude grade3 rate": rates[condition]["Claude"],
        "Both grade3 rate": rates[condition]["Both"],
        "Claude minus GPT": rates[condition]["Change"]
    })

rates_df = pd.DataFrame(rate_rows)

contrast_rows = []

for comparison, values in contrasts.items():

    contrast_rows.append({
        "Comparison": comparison,
        "GPT difference": values["GPT"],
        "Claude difference": values["Claude"],
        "Assessor sensitivity": values["Change"]
    })

contrasts_df = pd.DataFrame(
    contrast_rows
)



strata = [
    group.reset_index(drop=True)
    for _, group in df.groupby(
        [
            "Condition",
            "GPT grade",
            "Brave rank"
        ],
        sort=True
    )
]

rng = np.random.default_rng(
    RANDOM_SEED
)

rate_boot = {
    condition: {
        metric: []
        for metric in [
            "GPT", "Claude", "Both", "Change"
        ]
    }
    for condition in CONDITIONS
}

contrast_boot = {
    comparison: {
        metric: []
        for metric in [
            "GPT", "Claude", "Change"
        ]
    }
    for comparison in contrasts
}



print("Starting stratified bootstrap...")
print("Repetitions:", BOOTSTRAP_REPS)

for repetition in range(BOOTSTRAP_REPS):

    sampled_parts = []

    for stratum in strata:

        indices = rng.integers(
            0,
            len(stratum),
            size=len(stratum)
        )

        sampled_parts.append(
            stratum.iloc[indices]
        )

    boot_df = pd.concat(
        sampled_parts,
        ignore_index=True
    )

    boot_rates = estimate_rates(
        boot_df
    )

    boot_contrasts = estimate_contrasts(
        boot_rates
    )

    for condition in CONDITIONS:

        for metric in rate_boot[condition]:

            rate_boot[
                condition
            ][metric].append(
                boot_rates[condition][metric]
            )

    for comparison in contrast_boot:

        for metric in contrast_boot[comparison]:

            contrast_boot[
                comparison
            ][metric].append(
                boot_contrasts[comparison][metric]
            )

    if (repetition + 1) % 250 == 0:

        print(
            "Completed:",
            repetition + 1,
            "/",
            BOOTSTRAP_REPS
        )



def confidence_interval(values):

    values = np.asarray(
        values,
        dtype=float
    )

    return np.percentile(
        values,
        [2.5, 97.5]
    )


ci_rows = []

for condition in CONDITIONS:

    for metric in rate_boot[condition]:

        lower, upper = confidence_interval(
            rate_boot[condition][metric]
        )

        ci_rows.append({
            "Type": "Rate",
            "Comparison": condition,
            "Metric": metric,
            "Estimate": rates[condition][metric],
            "CI lower": lower,
            "CI upper": upper
        })

for comparison in contrasts:

    for metric in contrast_boot[comparison]:

        lower, upper = confidence_interval(
            contrast_boot[comparison][metric]
        )

        ci_rows.append({
            "Type": "Contrast",
            "Comparison": comparison,
            "Metric": metric,
            "Estimate": contrasts[comparison][metric],
            "CI lower": lower,
            "CI upper": upper
        })

ci_df = pd.DataFrame(ci_rows)



direction_rows = []

for comparison, values in contrasts.items():

    gpt = values["GPT"]
    claude = values["Claude"]

    if gpt > 0 and claude > 0:
        direction = "Positive under both"

    elif gpt < 0 and claude < 0:
        direction = "Negative under both"

    elif gpt == 0 or claude == 0:
        direction = "At least one zero difference"

    else:
        direction = "Direction reverses"

    gpt_ci = confidence_interval(
        contrast_boot[comparison]["GPT"]
    )

    claude_ci = confidence_interval(
        contrast_boot[comparison]["Claude"]
    )

    def interval_status(interval):

        if interval[0] > 0:
            return "Entirely positive"

        if interval[1] < 0:
            return "Entirely negative"

        return "Includes zero"

    direction_rows.append({
        "Comparison": comparison,
        "GPT difference": gpt,
        "GPT CI lower": gpt_ci[0],
        "GPT CI upper": gpt_ci[1],
        "GPT interval": interval_status(gpt_ci),
        "Claude difference": claude,
        "Claude CI lower": claude_ci[0],
        "Claude CI upper": claude_ci[1],
        "Claude interval": interval_status(claude_ci),
        "Direction": direction
    })

direction_df = pd.DataFrame(
    direction_rows
)



notes = [
    (
        "Purpose",
        "Assess sensitivity of grade-3 relevance "
        "rates to the choice of LLM assessor."
    ),
    (
        "Population",
        "73,230 URL relevance judgments."
    ),
    (
        "Sample",
        "1,000 stratified matched URL judgments."
    ),
    (
        "Assessors",
        "GPT-4o-mini and Claude Sonnet 5.5."
    ),
    (
        "Grade-3 definition",
        "Highly relevant retrieved result."
    ),
    (
        "Primary estimand",
        "Sampling-weighted proportion of URLs "
        "assigned relevance grade 3."
    ),
    (
        "Sampling weights",
        "Inverse inclusion probability from "
        "the original sampling design."
    ),
    (
        "Assessor sensitivity",
        "Claude grade-3 rate minus GPT grade-3 rate."
    ),
    (
        "BRUSE contrasts",
        "Difference between BRUSE and each "
        "comparison condition, separately "
        "for each assessor."
    ),
    (
        "Bootstrap",
        "2,000 stratified paired bootstrap resamples."
    ),
    (
        "Bootstrap strata",
        "Condition x original GPT grade x Brave rank."
    ),
    (
        "Pair preservation",
        "Both assessor judgments remain together "
        "when each sampled URL is resampled."
    ),
    (
        "Confidence intervals",
        "Percentile 95% bootstrap confidence intervals."
    ),
    (
        "Dependence limitation",
        "URL-level bootstrap does not fully "
        "account for correlation among URLs "
        "belonging to the same query concept."
    ),
    (
        "Retrieval metric limitation",
        "URL-level grade-3 rates are not "
        "P@10, RR@10, or nDCG@10."
    ),
    (
        "Assessor limitation",
        "Cross-model comparisons do not "
        "establish ground-truth relevance."
    ),
    (
        "Interpretation",
        "Same directional point estimates "
        "do not establish statistically "
        "significant retrieval superiority."
    )
]

notes_df = pd.DataFrame(
    notes,
    columns=["Item", "Explanation"]
)



with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    rates_df.to_excel(
        writer,
        sheet_name="Grade3_Rates",
        index=False
    )

    contrasts_df.to_excel(
        writer,
        sheet_name="BRUSE_Contrasts",
        index=False
    )

    ci_df.to_excel(
        writer,
        sheet_name="Bootstrap_CI",
        index=False
    )

    direction_df.to_excel(
        writer,
        sheet_name="Directional_Sensitivity",
        index=False
    )

    df.to_excel(
        writer,
        sheet_name="Matched_1000",
        index=False
    )

    notes_df.to_excel(
        writer,
        sheet_name="Methodology_Notes",
        index=False
    )



    for sheet in writer.book.worksheets:

        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

        for cell in sheet[1]:

            cell.font = Font(
                bold=True,
                color="FFFFFF"
            )

            cell.fill = PatternFill(
                fill_type="solid",
                fgColor="17365D"
            )

            cell.alignment = Alignment(
                wrap_text=True,
                vertical="center"
            )

        sheet.row_dimensions[1].height = 32

        for column in sheet.columns:

            letter = get_column_letter(
                column[0].column
            )

            max_length = max(
                len(str(cell.value))
                if cell.value is not None
                else 0
                for cell in column[:100]
            )

            sheet.column_dimensions[
                letter
            ].width = min(
                max(max_length + 3, 14),
                55
            )


    for sheet_name, columns in [
        ("Grade3_Rates", ["D", "E", "F", "G"]),
        ("BRUSE_Contrasts", ["B", "C", "D"]),
        ("Bootstrap_CI", ["D", "E", "F"]),
        (
            "Directional_Sensitivity",
            ["B", "C", "D", "F", "G", "H"]
        )
    ]:

        sheet = writer.book[sheet_name]

        for column in columns:

            for cell in sheet[column][1:]:
                cell.number_format = "0.00%"



print("\n" + "=" * 65)
print("GRADE-3 SENSITIVITY ANALYSIS COMPLETED")
print("=" * 65)

print("\nGRADE-3 RELEVANCE RATES:")
print(
    rates_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nBRUSE COMPARISONS:")
print(
    contrasts_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nDIRECTIONAL SENSITIVITY:")
print(
    direction_df.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")
print(OUTPUT_FILE)

print("\nMatched records:", len(df))
print("Bootstrap repetitions:", BOOTSTRAP_REPS)
print("Original workbook: UNCHANGED")


In [ ]:
### Cell_7 overall, analysis cell_3

import os
import numpy as np
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter



BASE_DIR = (
    r"D:\BORUE"
    )

INPUT_FILE = os.path.join(BASE_DIR,"Claude_GPT_Grade3_Sensitivity_Analysis.xlsx")

OUTPUT_FILE = os.path.join(BASE_DIR,"Claude_GPT_Final_Validation.xlsx")

REPS = 3000
SEED = 20261114

CONDITIONS = ["Common", "BRUSE", "Qwen"]

# ============================================================
# 1. LOAD AND VALIDATE
# ============================================================

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Matched_1000",
    dtype={"Sample ID": str}
)

required = [
    "Sample ID", "Concept", "Condition",
    "Brave rank", "GPT grade",
    "Claude grade", "Analysis weight"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

if len(df) != 1000:
    raise ValueError("Expected 1000 matched judgments.")

if df["Sample ID"].isna().any():
    raise ValueError("Missing sample IDs.")

if df["Sample ID"].duplicated().any():
    raise ValueError("Duplicate sample IDs.")

if df["Concept"].isna().any():
    raise ValueError("Missing concepts.")

df["Condition"] = df["Condition"].replace({
    "bruse_query2": "BRUSE"
})

if set(df["Condition"]) != set(CONDITIONS):
    raise ValueError("Unexpected retrieval conditions.")

for col in [
    "GPT grade", "Claude grade",
    "Brave rank", "Analysis weight"
]:
    df[col] = pd.to_numeric(df[col], errors="raise")

for col in ["GPT grade", "Claude grade"]:
    if not df[col].isin([0, 1, 2, 3]).all():
        raise ValueError(f"Invalid grades: {col}")

if not df["Brave rank"].between(1, 10).all():
    raise ValueError("Invalid Brave ranks.")

if (
    df["Analysis weight"].isna().any()
    or (df["Analysis weight"] <= 0).any()
):
    raise ValueError("Invalid sampling weights.")



n = len(df)

condition_codes = pd.Categorical(
    df["Condition"],
    categories=CONDITIONS
).codes

concept_codes, concept_names = pd.factorize(
    df["Concept"].astype(str),
    sort=True
)

n_concepts = len(concept_names)

gpt = (
    df["GPT grade"].to_numpy(dtype=int) == 3
).astype(float)

claude = (
    df["Claude grade"].to_numpy(dtype=int) == 3
).astype(float)

weights = df["Analysis weight"].to_numpy(
    dtype=float
)



stratum_codes, stratum_names = pd.factorize(
    list(zip(
        df["Condition"],
        df["GPT grade"],
        df["Brave rank"]
    )),
    sort=True
)

strata_indices = [
    np.flatnonzero(stratum_codes == k)
    for k in range(len(stratum_names))
]



def calculate_rates(row_weights):

    result = np.zeros((3, 2), dtype=float)

    for k in range(3):

        mask = condition_codes == k

        w = row_weights[mask]

        denominator = np.sum(w)

        if denominator <= 0:
            raise ValueError("Zero weighted denominator.")

        result[k, 0] = np.sum(
            w * gpt[mask]
        ) / denominator

        result[k, 1] = np.sum(
            w * claude[mask]
        ) / denominator

    return result


observed = calculate_rates(weights)



rng = np.random.default_rng(SEED)

boot_rates = np.zeros(
    (REPS, 3, 2),
    dtype=float
)

print("Starting concept-aware bootstrap...")
print("Unique concepts:", n_concepts)

for b in range(REPS):


    concept_multipliers = rng.exponential(
        scale=1.0,
        size=n_concepts
    )

    row_multipliers = concept_multipliers[
        concept_codes
    ].copy()



    for idx in strata_indices:

        original_total = weights[idx].sum()

        adjusted_total = np.sum(
            weights[idx] * row_multipliers[idx]
        )

        row_multipliers[idx] *= (
            original_total / adjusted_total
        )

    adjusted_weights = (
        weights * row_multipliers
    )

    boot_rates[b] = calculate_rates(
        adjusted_weights
    )

    if (b + 1) % 500 == 0:
        print(f"Completed {b + 1}/{REPS}")



def ci(values):

    return np.percentile(
        np.asarray(values, dtype=float),
        [2.5, 97.5]
    )


def direction_label(value, lower, upper):

    if lower > 0:
        return "Positive interval"

    if upper < 0:
        return "Negative interval"

    return "Interval includes zero"



rate_rows = []

for k, condition in enumerate(CONDITIONS):

    gpt_ci = ci(boot_rates[:, k, 0])
    claude_ci = ci(boot_rates[:, k, 1])

    rate_rows.append({
        "Condition": condition,
        "Sample n": int(
            np.sum(condition_codes == k)
        ),
        "GPT grade3 rate": observed[k, 0],
        "GPT CI lower": gpt_ci[0],
        "GPT CI upper": gpt_ci[1],
        "Claude grade3 rate": observed[k, 1],
        "Claude CI lower": claude_ci[0],
        "Claude CI upper": claude_ci[1]
    })

rates_df = pd.DataFrame(rate_rows)



comparison_pairs = [
    ("BRUSE - Common", 1, 0),
    ("BRUSE - Qwen", 1, 2)
]

comparison_rows = []

for label, a, b in comparison_pairs:

    gpt_diff = (
        observed[a, 0] - observed[b, 0]
    )

    claude_diff = (
        observed[a, 1] - observed[b, 1]
    )

    gpt_boot = (
        boot_rates[:, a, 0]
        - boot_rates[:, b, 0]
    )

    claude_boot = (
        boot_rates[:, a, 1]
        - boot_rates[:, b, 1]
    )

    gpt_lower, gpt_upper = ci(gpt_boot)
    claude_lower, claude_upper = ci(claude_boot)

    sensitivity = claude_diff - gpt_diff

    sensitivity_boot = (
        claude_boot - gpt_boot
    )

    sens_lower, sens_upper = ci(
        sensitivity_boot
    )

    comparison_rows.append({
        "Comparison": label,
        "GPT difference": gpt_diff,
        "GPT CI lower": gpt_lower,
        "GPT CI upper": gpt_upper,
        "GPT interpretation": direction_label(
            gpt_diff, gpt_lower, gpt_upper
        ),
        "Claude difference": claude_diff,
        "Claude CI lower": claude_lower,
        "Claude CI upper": claude_upper,
        "Claude interpretation": direction_label(
            claude_diff,
            claude_lower,
            claude_upper
        ),
        "Assessor sensitivity": sensitivity,
        "Sensitivity CI lower": sens_lower,
        "Sensitivity CI upper": sens_upper,
        "Same point-estimate direction": (
            bool(np.sign(gpt_diff) == np.sign(claude_diff))
        )
    })

comparison_df = pd.DataFrame(
    comparison_rows
)



def pct(value):
    return f"{100 * value:.2f}"


paper_rows = []

for k, condition in enumerate(CONDITIONS):

    gpt_lower, gpt_upper = ci(
        boot_rates[:, k, 0]
    )

    claude_lower, claude_upper = ci(
        boot_rates[:, k, 1]
    )

    paper_rows.append({
        "Condition": condition,
        "GPT grade3 (%)": pct(observed[k, 0]),
        "Claude grade3 (%)": pct(observed[k, 1]),
        "Claude 95% interval (%)": (
            f"{pct(claude_lower)}–"
            f"{pct(claude_upper)}"
        )
    })

paper_df = pd.DataFrame(paper_rows)



notes = pd.DataFrame([
    (
        "Purpose",
        "Final cross-model grade-3 relevance "
        "sensitivity analysis."
    ),
    (
        "Sample",
        "1000 matched URL judgments."
    ),
    (
        "Population",
        "73230 original URL judgments."
    ),
    (
        "Sampling",
        "Stratified by condition, original GPT "
        "grade and Brave rank."
    ),
    (
        "Primary measure",
        "Inverse-probability-weighted URL-level "
        "grade-3 relevance rate."
    ),
    (
        "Uncertainty method",
        "Shared concept-level exponential "
        "multipliers, normalized within "
        "original sampling strata."
    ),
    (
        "Replications",
        str(REPS)
    ),
    (
        "Random seed",
        str(SEED)
    ),
    (
        "GPT confidence intervals",
        "May have zero width because GPT grades "
        "defined the original sampling strata "
        "and stratum totals are fixed."
    ),
    (
        "Claude confidence intervals",
        "Approximate conditional sensitivity "
        "intervals, accounting for shared "
        "concept multipliers."
    ),
    (
        "Important limitation",
        "This is not an exact design-based "
        "cluster variance estimator. Strata "
        "with very small samples may not "
        "contribute adequately to uncertainty."
    ),
    (
        "Inference limitation",
        "Interval exclusion of zero is not "
        "proof of retrieval superiority."
    ),
    (
        "Retrieval limitation",
        "URL-level grade-3 rates are not "
        "P@10, RR@10 or nDCG@10."
    ),
    (
        "Ground truth",
        "Neither model is treated as a "
        "human-verified relevance standard."
    ),
    (
        "Final use",
        "Support a concise validation and "
        "limitations subsection in the paper."
    )
], columns=["Item", "Explanation"])



with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    rates_df.to_excel(
        writer,
        sheet_name="Grade3_Rates",
        index=False
    )

    comparison_df.to_excel(
        writer,
        sheet_name="Final_Comparisons",
        index=False
    )

    paper_df.to_excel(
        writer,
        sheet_name="Paper_Table",
        index=False
    )

    notes.to_excel(
        writer,
        sheet_name="Methodology_Notes",
        index=False
    )

    for sheet in writer.book.worksheets:

        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

        for cell in sheet[1]:

            cell.font = Font(
                bold=True,
                color="FFFFFF"
            )

            cell.fill = PatternFill(
                fill_type="solid",
                fgColor="17365D"
            )

            cell.alignment = Alignment(
                wrap_text=True,
                vertical="center"
            )

        sheet.row_dimensions[1].height = 30

        for column in sheet.columns:

            letter = get_column_letter(
                column[0].column
            )

            max_length = max(
                len(str(cell.value))
                if cell.value is not None
                else 0
                for cell in column[:100]
            )

            sheet.column_dimensions[
                letter
            ].width = min(
                max(max_length + 3, 14),
                55
            )

    for sheet_name, columns in [
        (
            "Grade3_Rates",
            ["C", "D", "E", "F", "G", "H"]
        ),
        (
            "Final_Comparisons",
            [
                "B", "C", "D",
                "F", "G", "H",
                "J", "K", "L"
            ]
        )
    ]:

        sheet = writer.book[sheet_name]

        for col in columns:
            for cell in sheet[col][1:]:
                cell.number_format = "0.00%"



print("\n" + "=" * 65)
print("FINAL VALIDATION ANALYSIS COMPLETED")
print("=" * 65)

print("\nGRADE-3 RATES:")
print(
    rates_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nBRUSE COMPARISONS:")
print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nPAPER TABLE:")
print(
    paper_df.to_string(index=False)
)

print("\nOUTPUT FILE:")
print(OUTPUT_FILE)

print("\nUnique concepts:", n_concepts)
print("Matched judgments:", n)
print("Bootstrap replications:", REPS)
print("Original workbook: UNCHANGED")


In [ ]:
### Cell_8 overall, Paper ready graph generartion

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator



BASE_DIR = (r"D:\BORUE")

FILE1 = os.path.join(BASE_DIR,"Claude_GPT_Agreement_Analysis.xlsx")

FILE2 = os.path.join(BASE_DIR,"Claude_GPT_Grade3_Sensitivity_Analysis.xlsx")

FILE3 = os.path.join(BASE_DIR,"Claude_GPT_Final_Validation.xlsx")

for path in [FILE1, FILE2, FILE3]:
    if not os.path.isfile(path):
        raise FileNotFoundError(path)


GREY   = "#75849A"
TEAL   = "#168396"
ORANGE = "#DA9C45"

DARK = "#202020"
GRID = "#E8ECF0"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 9,
    "axes.edgecolor": "#B8C2CC",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

def save_figure(fig, filename):
    png = os.path.join(BASE_DIR, filename + ".png")
    pdf = os.path.join(BASE_DIR, filename + ".pdf")

    fig.savefig(
        png,
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.18
    )

    fig.savefig(
        pdf,
        bbox_inches="tight",
        pad_inches=0.18
    )

    print("Saved:", png)
    print("Saved:", pdf)

def label_bars(ax, bars, suffix="%", decimals=2):
    for bar in bars:
        value = bar.get_height()
        ax.annotate(
            f"{value:.{decimals}f}{suffix}",
            (
                bar.get_x() + bar.get_width() / 2,
                value
            ),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=DARK
        )



summary = pd.read_excel(
    FILE1,
    sheet_name="Summary"
)

overall = summary.loc[
    summary["Condition"] == "All"
]

if len(overall) != 1:
    raise ValueError("Expected one All row in Summary.")

overall = overall.iloc[0]

agreement_labels = [
    "Exact grade\nagreement",
    "Grade-3 binary\nagreement",
    "Grade-3 positive\nagreement"
]

agreement_values = [
    float(overall["Weighted exact"]) * 100,
    float(overall["Weighted grade3 binary"]) * 100,
    float(overall["Weighted grade3 positive"]) * 100
]

fig, ax = plt.subplots(figsize=(8.2, 4.3))

x = np.arange(3)

bars = ax.bar(
    x,
    agreement_values,
    width=0.48,
    color=[GREY, TEAL, ORANGE]
)

ax.set_xticks(x, agreement_labels)
ax.set_ylim(0, 100)
ax.set_ylabel("Sampling-weighted agreement (%)")
ax.yaxis.set_major_locator(MultipleLocator(20))
ax.grid(axis="y", color=GRID, linewidth=0.8)

label_bars(ax, bars)

ax.set_title(
    "Independent Cross-Model Relevance Agreement",
    fontsize=12,
    fontweight="bold",
    pad=16
)

ax.text(
    0.5, 1.01,
    "1,000 reassessed URLs | GPT-4o-mini vs. Claude Sonnet 5.5",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=9,
    color="#666666"
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_1_Cross_Model_Agreement"
)

plt.show()
plt.close(fig)



rates = pd.read_excel(
    FILE2,
    sheet_name="Grade3_Rates"
)

rates = rates.set_index("Condition").loc[
    ["Common", "BRUSE", "Qwen"]
]

gpt_rates = (
    rates["GPT grade3 rate"].to_numpy(dtype=float)
    * 100
)

claude_rates = (
    rates["Claude grade3 rate"].to_numpy(dtype=float)
    * 100
)

fig, ax = plt.subplots(figsize=(8.2, 4.3))

x = np.arange(3)
width = 0.31

bars1 = ax.bar(
    x - width / 2,
    gpt_rates,
    width,
    color=ORANGE,
    label="GPT-4o-mini"
)

bars2 = ax.bar(
    x + width / 2,
    claude_rates,
    width,
    color=TEAL,
    label="Claude Sonnet 5.5"
)

ax.set_xticks(x, ["Common", "BRUSE", "Qwen"])
ax.set_ylim(0, max(gpt_rates.max(), claude_rates.max()) + 13)
ax.set_ylabel("Highly relevant retrieved URLs (%)")
ax.yaxis.set_major_locator(MultipleLocator(10))
ax.grid(axis="y", color=GRID, linewidth=0.8)

label_bars(ax, bars1)
label_bars(ax, bars2)

ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.16),
    ncol=2
)

ax.set_title(
    "Sensitivity of Grade-3 Relevance to the Assessor",
    fontsize=12,
    fontweight="bold",
    pad=40
)

ax.text(
    0.5, 1.02,
    "Sampling-weighted URL-level relevance rates",
    transform=ax.transAxes,
    ha="center",
    fontsize=9,
    color="#666666"
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_2_Grade3_Relevance_Sensitivity"
)

plt.show()
plt.close(fig)



comparisons = pd.read_excel(
    FILE3,
    sheet_name="Final_Comparisons"
)

comparisons = comparisons.set_index("Comparison").loc[
    ["BRUSE - Common", "BRUSE - Qwen"]
]

gpt_diff = (
    comparisons["GPT difference"].to_numpy(dtype=float)
    * 100
)

claude_diff = (
    comparisons["Claude difference"].to_numpy(dtype=float)
    * 100
)

claude_low = (
    comparisons["Claude CI lower"].to_numpy(dtype=float)
    * 100
)

claude_high = (
    comparisons["Claude CI upper"].to_numpy(dtype=float)
    * 100
)

# Asymmetric uncertainty around Claude point estimates
claude_errors = np.vstack([
    claude_diff - claude_low,
    claude_high - claude_diff
])

fig, ax = plt.subplots(figsize=(8.2, 4.5))

x = np.arange(2)
width = 0.30

bars1 = ax.bar(
    x - width / 2,
    gpt_diff,
    width,
    color=GREY,
    label="GPT-4o-mini"
)

bars2 = ax.bar(
    x + width / 2,
    claude_diff,
    width,
    color=ORANGE,
    label="Claude Sonnet 5.5"
)

ax.errorbar(
    x + width / 2,
    claude_diff,
    yerr=claude_errors,
    fmt="none",
    ecolor=DARK,
    elinewidth=1.2,
    capsize=5,
    capthick=1.2
)

ax.axhline(
    0,
    color="#555555",
    linewidth=1.0
)

ax.set_xticks(
    x,
    ["BRUSE − Common", "BRUSE − Qwen"]
)

ax.set_ylabel(
    "Difference in grade-3 relevance (percentage points)"
)

lower_limit = min(
    -5,
    float(np.min(gpt_diff)) - 5,
    float(np.min(claude_low)) - 5
)

upper_limit = max(
    25,
    float(np.max(claude_high)) + 5
)

ax.set_ylim(lower_limit, upper_limit)
ax.yaxis.set_major_locator(MultipleLocator(5))
ax.grid(axis="y", color=GRID, linewidth=0.8)

for bars in [bars1, bars2]:
    for bar in bars:
        value = bar.get_height()

        ax.annotate(
            f"{value:+.2f} pp",
            (
                bar.get_x() + bar.get_width() / 2,
                value
            ),
            xytext=(
                0,
                7 if value >= 0 else -15
            ),
            textcoords="offset points",
            ha="center",
            va="bottom" if value >= 0 else "top",
            fontsize=9,
            fontweight="bold",
            color=DARK
        )

ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.15),
    ncol=2
)

ax.set_title(
    "BRUSE Comparisons Across Relevance Assessors",
    fontsize=12,
    fontweight="bold",
    pad=38
)

ax.text(
    0.5, 1.02,
    "Error bars: approximate concept-aware 95% intervals "
    "(Claude only)",
    transform=ax.transAxes,
    ha="center",
    fontsize=8.5,
    color="#666666"
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_3_BRUSE_Final_Comparisons"
)

plt.show()
plt.close(fig)



print("\n" + "=" * 60)
print("ALL THREE PUBLICATION FIGURES GENERATED")
print("=" * 60)

print("Output directory:", BASE_DIR)
print("Figures:", 3)
print("Formats: PNG (600 DPI) + vector PDF")
print("Total files:", 6)
print("Original Excel workbooks: UNCHANGED")


In [ ]:
### Cell_9

import os
import pandas as pd
import matplotlib.pyplot as plt



BASE_DIR = (r"D:\BORUE")

INPUT_FILE = os.path.join(
    BASE_DIR, "Claude_GPT_Agreement_Analysis.xlsx"
)

OUTPUT_XLSX = os.path.join(
    BASE_DIR, "Cross_Model_Agreement_Table.xlsx"
)

OUTPUT_PNG = os.path.join(
    BASE_DIR, "Cross_Model_Agreement_Table.png"
)



df = pd.read_excel(INPUT_FILE, sheet_name="Summary")
df = df.set_index("Condition")

conditions = ["All", "Common", "BRUSE", "Qwen"]

if not set(conditions).issubset(df.index):
    raise ValueError("Required conditions are missing.")



metrics = [
    ("Exact agreement (%)", "Weighted exact", 100),
    ("Linear-weighted Cohen's κ", "Weighted linear kappa", 1),
    ("Quadratic-weighted Cohen's κ", "Weighted quadratic kappa", 1),
    ("Grade-3 positive agreement (%)",
     "Weighted grade3 positive", 100)
]

table_rows = []

for label, column, multiplier in metrics:

    values = [
        float(df.loc[c, column]) * multiplier
        for c in conditions
    ]

    table_rows.append([
        label,
        *[f"{v:.2f}" if multiplier == 100
          else f"{v:.3f}" for v in values]
    ])

columns = ["Metric", "Overall", "Common", "BRUSE", "Qwen"]

table_df = pd.DataFrame(table_rows, columns=columns)



with pd.ExcelWriter(
    OUTPUT_XLSX, engine="openpyxl"
) as writer:

    table_df.to_excel(
        writer,
        sheet_name="Agreement_Statistics",
        index=False
    )

    ws = writer.book.active
    ws.freeze_panes = "B2"

    from openpyxl.styles import Font, PatternFill, Alignment

    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(
            "solid", fgColor="17365D"
        )
        cell.alignment = Alignment(horizontal="center")

    ws.column_dimensions["A"].width = 39

    for col in ["B", "C", "D", "E"]:
        ws.column_dimensions[col].width = 17



fig, ax = plt.subplots(figsize=(10, 2.8))
ax.axis("off")

colors = [
    "#75849A",  # Grey
    "#168396",  # Teal
    "#DA9C45"   # Orange
]

table = ax.table(
    cellText=table_df.values,
    colLabels=table_df.columns,
    cellLoc="center",
    colLoc="center",
    colWidths=[0.39, 0.15, 0.15, 0.15, 0.16],
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.7)

for (row, col), cell in table.get_celld().items():

    cell.set_edgecolor("#D8DEE5")
    cell.set_linewidth(0.6)

    if row == 0:
        cell.set_facecolor("#17365D")
        cell.set_text_props(
            color="white", weight="bold"
        )

    elif row % 2 == 0:
        cell.set_facecolor("#F2F5F8")

    else:
        cell.set_facecolor("white")

    if row > 0 and col == 0:
        cell.set_text_props(ha="left")

    if row > 0 and col == 3:
        cell.set_text_props(
            color=colors[1], weight="bold"
        )

ax.set_title(
    "Cross-Model Relevance Agreement Statistics",
    fontsize=12,
    fontweight="bold",
    pad=15
)

plt.savefig(
    OUTPUT_PNG,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.15,
    facecolor="white"
)

plt.show()
plt.close(fig)

print("\nTABLE GENERATED SUCCESSFULLY")
print(table_df.to_string(index=False))
print("\nExcel:", OUTPUT_XLSX)
print("PNG:", OUTPUT_PNG)
